<a href="https://colab.research.google.com/github/Namita-Muduli/Recommendation-System/blob/main/Improve_Base_Paper.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Improving on the base paper (same dataset, same split)
**Base paper:** Almazroei, E. E. (2026). *Scientific Reports, 16*, Article 19506. https://doi.org/10.1038/s41598-026-49894-1

This notebook re-runs the author's exact baseline and then tries to improve on it on the **same OULAD data and the same stratified 80/20 split (`random_state=42`)**.

**Fairness rules**
* Every choice (SMOTE or not, hyper-parameters, ensemble, decision threshold) is made by cross-validation on the **training set only**. The 20% test set is only scored afterwards.
* The author's baseline (B0) is re-run first, so the comparison is like for like. B0 should print a test accuracy close to your earlier run (about 0.9529).
* New features are ordinary behavioural / assessment summaries from the raw OULAD tables. **No exam marks and no extra features derived from the unregistration date** are added.
* A *clean* variant (C0, C1) without the three unregistration-based features is also reported, so the gain is not only a leakage effect.
* Everything is reported, including configurations that do not help. Do not change seeds or the split to make a number look better.

**How to run in Google Colab:** run the cells from top to bottom. Expect roughly 45 to 90 minutes on the real data (the click file has about 10 million rows). Keep the browser tab open.

## 0. Install packages and get the data (Colab)
Run this once per session. If Colab asks you to restart the session after the install, do that and run this cell again.

In [ ]:
!pip -q install xgboost lightgbm catboost imbalanced-learn

In [ ]:
# Mount Google Drive and copy the 7 OULAD CSV files to Colab's fast local disk.
# Change DRIVE_FOLDER if your files are in a different Drive folder.
import os
DRIVE_FOLDER = "/content/drive/MyDrive/OULAD/OULAD"
from google.colab import drive
drive.mount('/content/drive')
os.makedirs('/content/oulad_data', exist_ok=True)
!cp "$DRIVE_FOLDER"/*.csv /content/oulad_data/
!ls -lh /content/oulad_data
# You should see 7 files; studentVle.csv is about 433 MB.

Mounted at /content/drive
total 443M
-rw------- 1 root root 8.1K Nov 17 10:43 assessments.csv
-rw------- 1 root root  526 Nov 17 10:43 courses.csv
-rw------- 1 root root 5.5M Nov 17 10:43 studentAssessment.csv
-rw------- 1 root root 3.4M Nov 17 10:43 studentInfo.csv
-rw------- 1 root root 1.1M Nov 17 10:43 studentRegistration.csv
-rw------- 1 root root 433M Nov 17 10:44 studentVle.csv
-rw------- 1 root root 255K Nov 17 10:44 vle.csv


## 1. Settings

In [ ]:
from types import SimpleNamespace
ARGS = SimpleNamespace(
    data_dir="/content/oulad_data",              # folder with the 7 OULAD CSV files
    out="/content/drive/MyDrive/out_improve",    # results are saved here (Drive keeps them if the session ends)
    n_iter=25,                                   # random-search candidates for XGBoost
    quick=False,                                 # True = tiny settings, only for checking that the code runs
)

In [ ]:
import json, os, sys, time, warnings
from types import SimpleNamespace
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy import stats

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, cross_val_predict, RandomizedSearchCV
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier, VotingClassifier

try:
    from xgboost import XGBClassifier
except Exception:
    XGBClassifier = None
try:
    from lightgbm import LGBMClassifier
except Exception:
    LGBMClassifier = None
try:
    from catboost import CatBoostClassifier
except Exception:
    CatBoostClassifier = None
try:
    from imblearn.over_sampling import SMOTE
    from imblearn.pipeline import Pipeline as ImbPipeline
except Exception:
    SMOTE = None
    ImbPipeline = None

In [ ]:
if ARGS.quick:
    ARGS.n_iter = 3
os.makedirs(ARGS.out, exist_ok=True)
T0 = time.time()
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
def log(m): print(f"[{time.time()-T0:6.0f}s] {m}", flush=True)
log(f"xgboost={'yes' if XGBClassifier else 'NO (fallback: sklearn HistGradientBoosting)'}  lightgbm={'yes' if LGBMClassifier else 'no'}  "
    f"catboost={'yes' if CatBoostClassifier else 'no'}  smote={'yes' if SMOTE else 'NO'}")

[     0s] xgboost=yes  lightgbm=yes  catboost=yes  smote=yes


## 2. The author's data preparation (copied from the original notebook, unchanged except the data folder)

### Load the 7 CSV files

In [ ]:
# ---------- author's cell: load ----------
# ============================================================
# 2. Load OULAD CSV Files
#    Put all 7 CSVs in the same folder as this notebook
#    or change DATA_DIR accordingly.
# ============================================================

DATA_DIR = ARGS.data_dir

paths = {
    "assessments":         os.path.join(DATA_DIR, "assessments.csv"),
    "courses":             os.path.join(DATA_DIR, "courses.csv"),
    "vle":                 os.path.join(DATA_DIR, "vle.csv"),
    "studentRegistration": os.path.join(DATA_DIR, "studentRegistration.csv"),
    "studentInfo":         os.path.join(DATA_DIR, "studentInfo.csv"),
    "studentAssessment":   os.path.join(DATA_DIR, "studentAssessment.csv"),
    "studentVle":          os.path.join(DATA_DIR, "studentVle.csv"),
}

for name, p in paths.items():
    if not os.path.exists(p):
        raise FileNotFoundError(f"{name} file not found at: {p}")

df_assessments         = pd.read_csv(paths["assessments"])
df_courses             = pd.read_csv(paths["courses"])
df_vle                 = pd.read_csv(paths["vle"])
df_student_registration= pd.read_csv(paths["studentRegistration"])
df_student_info        = pd.read_csv(paths["studentInfo"])
df_student_assessment  = pd.read_csv(paths["studentAssessment"])
df_student_vle         = pd.read_csv(paths["studentVle"])

print("All CSV files loaded.")
print("assessments:", df_assessments.shape)
print("courses:", df_courses.shape)
print("vle:", df_vle.shape)
print("studentRegistration:", df_student_registration.shape)
print("studentInfo:", df_student_info.shape)
print("studentAssessment:", df_student_assessment.shape)
print("studentVle:", df_student_vle.shape)

All CSV files loaded.
assessments: (206, 6)
courses: (22, 3)
vle: (6364, 6)
studentRegistration: (32593, 5)
studentInfo: (32593, 12)
studentAssessment: (173912, 5)
studentVle: (10655280, 6)


### Base student dataset and target

In [ ]:
# ---------- author's cell: base dataset + target ----------
# ============================================================
# 3. Base Student Dataset + Target Variable
#    Merge studentRegistration + studentInfo + courses
# ============================================================

# Merge registration with student info
df_students = pd.merge(
    df_student_registration,
    df_student_info,
    on=["code_module", "code_presentation", "id_student"],
    how="inner"
)

# Merge course length
df_students = pd.merge(
    df_students,
    df_courses,
    on=["code_module", "code_presentation"],
    how="left"
)

print("Base student-level dataset shape:", df_students.shape)

# Create binary target: passed (Pass/Distinction = 1, Fail/Withdrawn = 0)
df_students["passed"] = df_students["final_result"].apply(
    lambda x: 1 if x in ["Pass", "Distinction"] else 0
)

print(df_students["final_result"].value_counts())
print("\nTarget distribution (Successful):")
print(df_students["passed"].value_counts(normalize=True))

# Optional: basic time-based feature
# If date_unregistration is NaN, assume student remained to end of module
df_students["date_unregistration_filled"] = df_students["date_unregistration"].fillna(
    df_students["module_presentation_length"]
)
df_students["enrolment_length"] = (
    df_students["date_unregistration_filled"] - df_students["date_registration"]
)

Base student-level dataset shape: (32593, 15)
final_result
Pass           12361
Withdrawn      10156
Fail            7052
Distinction     3024
Name: count, dtype: int64

Target distribution (Successful):
passed
0    0.527966
1    0.472034
Name: proportion, dtype: float64


### Assessment features (reconstructed cell, matches the author's saved shape (25843, 14))

In [ ]:
# ---------- author's cell: assessment features (reconstructed cell) ----------
# ============================================================
# 4. Assessment Features (Aggregated per Student-Module)
# ------------------------------------------------------------
# RECONSTRUCTED CELL. In the shared notebook this cell was empty and
# `group_cols` / `df_agg_assess` were never defined, so cell 6 and 7
# fail with NameError. This code rebuilds the same 14 columns that the
# author's saved output shows (shape (25843, 14)).
# ============================================================

group_cols = ["id_student", "code_module", "code_presentation"]

df_assess_merged = pd.merge(
    df_student_assessment,
    df_assessments,
    on="id_assessment",
    how="left"
)
df_assess_merged["weighted_score"] = df_assess_merged["score"] * df_assess_merged["weight"]

df_agg_assess = df_assess_merged.groupby(group_cols).agg(
    num_assessments=("id_assessment", "count"),
    mean_score=("score", "mean"),
    max_score=("score", "max"),
    min_score=("score", "min"),
    std_score=("score", "std"),
    total_weight=("weight", "sum"),
    total_weighted_score=("weighted_score", "sum")
).reset_index()

df_agg_assess["weighted_avg_score"] = (
    df_agg_assess["total_weighted_score"] / df_agg_assess["total_weight"]
)

# number of TMA / CMA / Exam submissions per student-module
type_counts = df_assess_merged.pivot_table(
    index=group_cols,
    columns="assessment_type",
    values="id_assessment",
    aggfunc="count",
    fill_value=0
)
type_counts = type_counts.reindex(columns=["TMA", "CMA", "Exam"], fill_value=0)
type_counts.columns = [f"num_{c}" for c in type_counts.columns]
type_counts = type_counts.reset_index()

df_agg_assess = pd.merge(df_agg_assess, type_counts, on=group_cols, how="left")

print("Assessment features shape:", df_agg_assess.shape)
print("(The author's saved output for this cell was: (25843, 14))")

Assessment features shape: (25843, 14)
(The author's saved output for this cell was: (25843, 14))


### VLE features

In [ ]:
# ---------- author's cell: VLE features ----------
# ============================================================
# 5. VLE Features (Aggregated per Student-Module)
# ============================================================

# Merge studentVle with vle to get activity_type
df_svle = pd.merge(
    df_student_vle,
    df_vle[["id_site", "activity_type"]],
    on="id_site",
    how="left"
)

# Aggregate main VLE usage features
df_agg_vle = df_svle.groupby(group_cols).agg(
    total_clicks=("sum_click", "sum"),
    days_active=("date", "nunique"),
    num_resources=("id_site", "nunique")
).reset_index()

# Optional: clicks by activity_type (pivot)
activity_clicks = df_svle.pivot_table(
    index=group_cols,
    columns="activity_type",
    values="sum_click",
    aggfunc="sum",
    fill_value=0
)

activity_clicks.columns = [f"clicks_{c}" for c in activity_clicks.columns]
activity_clicks = activity_clicks.reset_index()

# Merge aggregated VLE features
df_agg_vle = pd.merge(df_agg_vle, activity_clicks, on=group_cols, how="left")

print("VLE features shape:", df_agg_vle.shape)

VLE features shape: (29228, 26)


### Final modelling dataset

In [ ]:
# ---------- author's cell: final dataset ----------
# ============================================================
# 6. Build Final Modeling Dataset
#    Merge base student info + assessment + vle features
# ============================================================

df_final = df_students.merge(
    df_agg_assess,
    on=["id_student", "code_module", "code_presentation"],
    how="left"
)

df_final = df_final.merge(
    df_agg_vle,
    on=["id_student", "code_module", "code_presentation"],
    how="left"
)

print("Final dataset shape (before cleaning):", df_final.shape)

# Basic missing value handling:
# numeric feature NaNs -> 0
numeric_cols = df_final.select_dtypes(include=["int64", "float64", "int32", "float32"]).columns.tolist()
# but keep target separate
numeric_cols = [c for c in numeric_cols if c not in ["passed"]]

df_final[numeric_cols] = df_final[numeric_cols].fillna(0)

# Some date columns may still be NaN, handle them safely
for col in ["date_unregistration", "date_unregistration_filled", "enrolment_length"]:
    if col in df_final.columns:
        df_final[col] = df_final[col].fillna(0)

print("Final dataset after basic NA handling:")
df_final.info()

Final dataset shape (before cleaning): (32593, 52)
Final dataset after basic NA handling:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32593 entries, 0 to 32592
Data columns (total 52 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   code_module                 32593 non-null  object 
 1   code_presentation           32593 non-null  object 
 2   id_student                  32593 non-null  int64  
 3   date_registration           32593 non-null  float64
 4   date_unregistration         32593 non-null  float64
 5   gender                      32593 non-null  object 
 6   region                      32593 non-null  object 
 7   highest_education           32593 non-null  object 
 8   imd_band                    31482 non-null  object 
 9   age_band                    32593 non-null  object 
 10  num_of_prev_attempts        32593 non-null  int64  
 11  studied_credits             32593 non-null  int64  
 12

### Features, target and the author's train/test split

In [ ]:
# ---------- author's cell: features, target, split ----------
# ============================================================
# 7. Select Features & Target for Modeling
# ============================================================

# Target:
TARGET_COL = "passed"

# Columns to drop from features (IDs and strong leak targets)
drop_cols = [
    "passed",
    "final_result",
    "id_student",
    # If you think some of these leak too much future info, you can also drop them:
    # "date_unregistration", "date_unregistration_filled"
]

X = df_final.drop(columns=[c for c in drop_cols if c in df_final.columns])
y = df_final[TARGET_COL].astype(int)

print("X shape:", X.shape)
print("y shape:", y.shape)

# Identify numeric and categorical feature columns
numeric_features = X.select_dtypes(include=["int64", "float64", "int32", "float32"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)

# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=RANDOM_STATE
)

print("Train:", X_train.shape, "Test:", X_test.shape)

X shape: (32593, 49)
y shape: (32593,)
Numeric features: ['date_registration', 'date_unregistration', 'num_of_prev_attempts', 'studied_credits', 'module_presentation_length', 'date_unregistration_filled', 'enrolment_length', 'num_assessments', 'mean_score', 'max_score', 'min_score', 'std_score', 'total_weight', 'total_weighted_score', 'weighted_avg_score', 'num_TMA', 'num_CMA', 'num_Exam', 'total_clicks', 'days_active', 'num_resources', 'clicks_dataplus', 'clicks_dualpane', 'clicks_externalquiz', 'clicks_folder', 'clicks_forumng', 'clicks_glossary', 'clicks_homepage', 'clicks_htmlactivity', 'clicks_oucollaborate', 'clicks_oucontent', 'clicks_ouelluminate', 'clicks_ouwiki', 'clicks_page', 'clicks_questionnaire', 'clicks_quiz', 'clicks_repeatactivity', 'clicks_resource', 'clicks_sharedsubpage', 'clicks_subpage', 'clicks_url']
Categorical features: ['code_module', 'code_presentation', 'gender', 'region', 'highest_education', 'imd_band', 'age_band', 'disability']
Train: (26074, 49) Test: (

## 3. Extra features
Ordinary behaviour and assessment summaries (timing and regularity of activity, late/missed work, score trend). No exam marks, nothing from the unregistration date.

In [ ]:
import gc
try:
    del df_student_vle          # the merged table df_svle is enough; frees about 1 GB of RAM before the heavy groupby steps
except NameError:
    pass
gc.collect()
log("building extra features")
KEYS = ["id_student", "code_module", "code_presentation"]
LEAKY_UNREG = ["date_unregistration", "date_unregistration_filled", "enrolment_length"]

def build_extra_features():
    # ---- VLE behaviour: per-day clicks, timing, regularity
    daily = df_svle.groupby(KEYS + ["date"], as_index=False)["sum_click"].sum()
    daily = daily.merge(df_courses[["code_module", "code_presentation", "module_presentation_length"]],
                        on=["code_module", "code_presentation"], how="left")
    L = daily["module_presentation_length"]
    daily["week"] = daily["date"] // 7
    daily["c_prestart"] = np.where(daily["date"] < 0, daily["sum_click"], 0)
    daily["c_first4w"] = np.where((daily["date"] >= 0) & (daily["date"] < 28), daily["sum_click"], 0)
    daily["c_last4w"] = np.where(daily["date"] > L - 28, daily["sum_click"], 0)
    daily["c_lastthird"] = np.where(daily["date"] > L * 2 / 3, daily["sum_click"], 0)
    daily = daily.sort_values(KEYS + ["date"])
    daily["gap"] = daily.groupby(KEYS)["date"].diff()
    g = daily.groupby(KEYS)
    v = g.agg(first_active_day=("date", "min"), last_active_day=("date", "max"), max_daily_clicks=("sum_click", "max"),
              mean_daily_clicks=("sum_click", "mean"), std_daily_clicks=("sum_click", "std"), max_gap_days=("gap", "max"),
              c_prestart=("c_prestart", "sum"), c_first4w=("c_first4w", "sum"), c_last4w=("c_last4w", "sum"),
              c_lastthird=("c_lastthird", "sum"), total_all=("sum_click", "sum"),
              module_len=("module_presentation_length", "first")).reset_index()
    v["n_active_weeks"] = daily.groupby(KEYS)["week"].nunique().values
    v["last_active_ratio"] = v["last_active_day"] / v["module_len"]
    v["days_inactive_at_end"] = v["module_len"] - v["last_active_day"]
    v["active_span"] = v["last_active_day"] - v["first_active_day"]
    for c, name in [("c_first4w", "share_first4w"), ("c_last4w", "share_last4w"), ("c_lastthird", "share_lastthird"), ("c_prestart", "share_prestart")]:
        v[name] = v[c] / v["total_all"].replace(0, np.nan)
    v = v.drop(columns=["c_prestart", "c_first4w", "c_last4w", "c_lastthird", "total_all", "module_len"])
    ntypes = df_svle.groupby(KEYS)["activity_type"].nunique().rename("n_activity_types").reset_index()
    v = v.merge(ntypes, on=KEYS, how="left")

    # ---- assessment behaviour: punctuality, completion, trend
    am = df_assess_merged.copy()
    am["late"] = am["date_submitted"] - am["date"]
    am["is_late"] = (am["late"] > 0).astype(int)
    am["below40"] = (am["score"] < 40).astype(int)
    a = am.groupby(KEYS).agg(n_late=("is_late", "sum"), mean_lateness=("late", "mean"), max_lateness=("late", "max"),
                             n_below40=("below40", "sum"), median_score=("score", "median"),
                             first_sub_day=("date_submitted", "min"), last_sub_day=("date_submitted", "max")).reset_index()
    fl = am.sort_values(KEYS + ["date_submitted"]).groupby(KEYS)["score"].agg(score_first="first", score_last="last").reset_index()
    fl["score_trend"] = fl["score_last"] - fl["score_first"]
    a = a.merge(fl, on=KEYS, how="left")
    nonexam = am[am["assessment_type"] != "Exam"].groupby(KEYS).size().rename("n_sub_nonexam").reset_index()
    due = df_assessments[df_assessments["assessment_type"] != "Exam"].groupby(["code_module", "code_presentation"]).size().rename("n_due").reset_index()
    a = a.merge(nonexam, on=KEYS, how="left")
    a = a.merge(due, on=["code_module", "code_presentation"], how="left")
    a["frac_submitted"] = a["n_sub_nonexam"] / a["n_due"].replace(0, np.nan)
    a["n_missed"] = (a["n_due"] - a["n_sub_nonexam"]).clip(lower=0)
    a["share_below40"] = a["n_below40"] / a["n_sub_nonexam"].replace(0, np.nan)
    return v.merge(a, on=KEYS, how="outer")

extra = build_extra_features()
extra_cols = [c for c in extra.columns if c not in KEYS]
aligned = df_final[KEYS].merge(extra, on=KEYS, how="left").drop(columns=KEYS)
assert len(aligned) == len(X), "row count changed - split would no longer match the author's"
aligned.index = X.index
for c in ["n_late", "n_below40", "n_missed", "n_activity_types", "n_active_weeks", "n_sub_nonexam"]:
    aligned[c] = aligned[c].fillna(0)
X2 = pd.concat([X, aligned], axis=1)
X2_train, X2_test = X2.loc[X_train.index], X2.loc[X_test.index]
num2 = numeric_features + extra_cols
cat2 = categorical_features
log(f"extra features: {len(extra_cols)} new columns -> {X2.shape[1]} total; same split rows as the author "
    f"(train {len(X2_train)}, test {len(X2_test)})")
pd.Series(extra_cols).to_csv(os.path.join(ARGS.out, "extra_features.csv"), index=False, header=["feature"])

[    28s] building extra features
[    39s] extra features: 30 new columns -> 79 total; same split rows as the author (train 26074, test 6519)


## 4. Models, pipelines and helper functions

In [ ]:
# =====================================================================================================
# PART 3: models, pipelines, experiments (all choices made by cross-validation on the training set)
# =====================================================================================================
cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv3 = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

def make_prep(num_cols, cat_cols, indicator):
    num_t = Pipeline([("imputer", SimpleImputer(strategy="median", add_indicator=indicator)), ("scaler", StandardScaler())])
    cat_t = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("onehot", OneHotEncoder(handle_unknown="ignore"))])
    return ColumnTransformer([("num", num_t, num_cols), ("cat", cat_t, cat_cols)])

AUTHOR_XGB = dict(n_estimators=300, learning_rate=0.05, max_depth=5, subsample=0.8, colsample_bytree=0.8)

def gbm(**p):
    if XGBClassifier is not None:
        q = dict(AUTHOR_XGB); q.update(p)
        return XGBClassifier(objective="binary:logistic", eval_metric="logloss", random_state=RANDOM_STATE, n_jobs=-1, **q)
    return HistGradientBoostingClassifier(max_iter=p.get("n_estimators", 300), learning_rate=p.get("learning_rate", 0.05),
                                          max_depth=p.get("max_depth", 5), random_state=RANDOM_STATE)

def make_pipe(model, num_cols, cat_cols, smote, indicator):
    prep = make_prep(num_cols, cat_cols, indicator)
    if smote and SMOTE is not None:
        return ImbPipeline([("preprocess", prep), ("smote", SMOTE(random_state=RANDOM_STATE)), ("model", model)])
    return Pipeline([("preprocess", prep), ("model", model)])

def cv_acc(pipe, Xtr):
    s = cross_val_score(pipe, Xtr, y_train, scoring="accuracy", cv=cv5, n_jobs=1)
    return float(s.mean()), float(s.std())

In [ ]:
results = []
test_preds = {}
def evaluate(name, pipe, Xtr, Xte, cv=None, note=""):
    t = time.time()
    if cv is None:
        cv = cv_acc(pipe, Xtr)
    pipe.fit(Xtr, y_train)
    p = pipe.predict_proba(Xte)[:, 1]
    yhat = (p >= 0.5).astype(int)
    r = {"config": name, "cv_acc": cv[0], "cv_acc_sd": cv[1], "test_acc": accuracy_score(y_test, yhat),
         "test_f1": f1_score(y_test, yhat), "test_auc": roc_auc_score(y_test, p),
         "test_errors": int((yhat != y_test.values).sum()), "note": note}
    results.append(r); test_preds[name] = (yhat, p)
    log(f"{name:<44} CV acc {r['cv_acc']:.4f}±{r['cv_acc_sd']:.4f} | TEST acc {r['test_acc']:.4f}  f1 {r['test_f1']:.4f}  auc {r['test_auc']:.4f}  ({time.time()-t:.0f}s)")
    pd.DataFrame(results).to_csv(os.path.join(ARGS.out, "improvement_results.csv"), index=False)
    return pipe

## 5. Experiments
Test accuracy is printed for every configuration, but the choice of the final model uses training-set cross-validation only.

### B0 and B1: the author's configuration, and the same without SMOTE

In [ ]:
# ---- B0: the author's exact configuration
B0 = evaluate("B0 author: XGBoost + SMOTE (author features)",
              make_pipe(gbm(), numeric_features, categorical_features, True, False), X_train, X_test)
# ---- B1: same, without SMOTE (the classes are nearly balanced: 47.2% vs 52.8%)
B1 = evaluate("B1 author features, no SMOTE",
              make_pipe(gbm(), numeric_features, categorical_features, False, False), X_train, X_test)
use_smote = (SMOTE is not None) and (results[0]["cv_acc"] >= results[1]["cv_acc"])
log(f"decision from training CV: use SMOTE for the improved models = {use_smote}")

[    79s] B0 author: XGBoost + SMOTE (author features) CV acc 0.9513±0.0030 | TEST acc 0.9529  f1 0.9514  auc 0.9883  (40s)
[   101s] B1 author features, no SMOTE                 CV acc 0.9506±0.0027 | TEST acc 0.9524  f1 0.9508  auc 0.9883  (22s)
[   101s] decision from training CV: use SMOTE for the improved models = True


### F1: extra features, author's XGBoost settings

In [ ]:
# ---- F1: extra features, author's XGBoost settings
F1 = evaluate("F1 + extra features (author XGB settings)",
              make_pipe(gbm(), num2, cat2, use_smote, True), X2_train, X2_test)

[   158s] F1 + extra features (author XGB settings)    CV acc 0.9589±0.0021 | TEST acc 0.9610  f1 0.9596  auc 0.9912  (57s)


### F2: tuned XGBoost (random search, 3-fold CV inside the training set)

In [ ]:
# ---- F2: tuned XGBoost on extra features (random search, 3-fold CV inside the training set)
log(f"random search: {ARGS.n_iter} candidates x 3 folds")
space = ({"model__n_estimators": [300, 500, 800], "model__max_depth": [3, 4, 5, 6, 7, 8], "model__learning_rate": [0.02, 0.03, 0.05, 0.08],
          "model__subsample": [0.7, 0.8, 0.9, 1.0], "model__colsample_bytree": [0.5, 0.7, 0.8, 1.0],
          "model__min_child_weight": [1, 3, 5, 7], "model__gamma": [0, 0.1, 0.3], "model__reg_lambda": [0.5, 1, 3, 5]}
         if XGBClassifier is not None else
         {"model__max_iter": [200, 300, 500], "model__max_depth": [3, 4, 5, 6, None], "model__learning_rate": [0.03, 0.05, 0.08],
          "model__min_samples_leaf": [10, 20, 40]})
rs = RandomizedSearchCV(make_pipe(gbm(), num2, cat2, use_smote, True), space, n_iter=ARGS.n_iter, scoring="accuracy",
                        cv=cv3, random_state=RANDOM_STATE, n_jobs=1, refit=False)
rs.fit(X2_train, y_train)
best_params = {k.replace("model__", ""): v for k, v in rs.best_params_.items()}
log(f"best params (CV on training set only): {best_params}  (3-fold acc {rs.best_score_:.4f})")
json.dump({k: (v.item() if hasattr(v, "item") else v) for k, v in best_params.items()}, open(os.path.join(ARGS.out, "best_params.json"), "w"), indent=2)
if XGBClassifier is not None:
    tuned = lambda: gbm(**best_params)
else:
    tuned = lambda: HistGradientBoostingClassifier(random_state=RANDOM_STATE, **{("max_iter" if k == "n_estimators" else k): v for k, v in best_params.items()})
F2 = evaluate("F2 + extra features + tuned XGBoost",
              make_pipe(tuned(), num2, cat2, use_smote, True), X2_train, X2_test)

[   158s] random search: 25 candidates x 3 folds
[  1122s] best params (CV on training set only): {'subsample': 0.8, 'reg_lambda': 1, 'n_estimators': 800, 'min_child_weight': 3, 'max_depth': 8, 'learning_rate': 0.02, 'gamma': 0, 'colsample_bytree': 0.8}  (3-fold acc 0.9584)
[  1323s] F2 + extra features + tuned XGBoost          CV acc 0.9593±0.0024 | TEST acc 0.9615  f1 0.9600  auc 0.9912  (201s)


### F3: soft-voting ensemble

In [ ]:
# ---- F3: soft-voting ensemble of tuned boosted trees + other tree models
members = [("gbm", tuned())]
if LGBMClassifier is not None:
    members.append(("lgbm", LGBMClassifier(n_estimators=600, learning_rate=0.03, num_leaves=31, subsample=0.8, subsample_freq=1,
                                           colsample_bytree=0.8, random_state=RANDOM_STATE, verbose=-1, n_jobs=-1)))
if CatBoostClassifier is not None and not ARGS.quick:
    members.append(("cat", CatBoostClassifier(iterations=600, depth=6, learning_rate=0.05, random_seed=RANDOM_STATE, verbose=0, thread_count=-1)))
members.append(("rf", RandomForestClassifier(n_estimators=400, min_samples_leaf=2, n_jobs=-1, random_state=RANDOM_STATE)))
F3 = evaluate("F3 + extra features + soft-voting ensemble (" + "+".join(m[0] for m in members) + ")",
              make_pipe(VotingClassifier(members, voting="soft"), num2, cat2, use_smote, True), X2_train, X2_test)

[  1926s] F3 + extra features + soft-voting ensemble (gbm+lgbm+cat+rf) CV acc 0.9592±0.0020 | TEST acc 0.9626  f1 0.9612  auc 0.9913  (602s)


### Final choice, F4 (threshold from out-of-fold predictions)

In [ ]:
# ---- final choice: best CV accuracy on TRAINING data (test set is not used for the choice)
cands = [r for r in results if r["config"].startswith("F")]
best = max(cands, key=lambda r: r["cv_acc"])
log(f"selected by training-CV accuracy: {best['config']}  (CV {best['cv_acc']:.4f})")
best_name = best["config"]
best_pipe_factory = {"F1": lambda: make_pipe(gbm(), num2, cat2, use_smote, True),
                     "F2": lambda: make_pipe(tuned(), num2, cat2, use_smote, True),
                     "F3": lambda: make_pipe(VotingClassifier(members, voting="soft"), num2, cat2, use_smote, True)}[best_name[:2]]

# ---- F4: decision threshold chosen from out-of-fold training predictions
log("threshold selection from out-of-fold training predictions")
oof = cross_val_predict(best_pipe_factory(), X2_train, y_train, cv=cv5, method="predict_proba", n_jobs=1)[:, 1]
grid = np.linspace(0.30, 0.70, 41)
thr = float(grid[int(np.argmax([accuracy_score(y_train, (oof >= t).astype(int)) for t in grid]))])
fin = best_pipe_factory().fit(X2_train, y_train)
pf = fin.predict_proba(X2_test)[:, 1]
yf = (pf >= thr).astype(int)
r = {"config": f"F4 = {best_name[:2]} + OOF-chosen threshold {thr:.2f}", "cv_acc": float(accuracy_score(y_train, (oof >= thr).astype(int))), "cv_acc_sd": np.nan,
     "test_acc": accuracy_score(y_test, yf), "test_f1": f1_score(y_test, yf), "test_auc": roc_auc_score(y_test, pf),
     "test_errors": int((yf != y_test.values).sum()), "note": "final candidate"}
results.append(r); test_preds[r["config"]] = (yf, pf)
log(f"{r['config']:<44} TEST acc {r['test_acc']:.4f} f1 {r['test_f1']:.4f} auc {r['test_auc']:.4f}")

[  1926s] selected by training-CV accuracy: F2 + extra features + tuned XGBoost  (CV 0.9593)
[  1926s] threshold selection from out-of-fold training predictions
[  2141s] F4 = F2 + OOF-chosen threshold 0.50          TEST acc 0.9615 f1 0.9600 auc 0.9912


### C0 and C1: clean variants without the unregistration features

In [ ]:
# ---- C: "clean" variants without the three unregistration-based features (honest view)
clean_num1 = [c for c in numeric_features if c not in LEAKY_UNREG]
clean_num2 = [c for c in num2 if c not in LEAKY_UNREG]
evaluate("C0 clean: author XGBoost, author features minus unregistration",
         make_pipe(gbm(), clean_num1, categorical_features, True, False), X_train, X_test, cv=(np.nan, np.nan), note="no unregistration features")
evaluate("C1 clean: best improved model minus unregistration",
         make_pipe(tuned(), clean_num2, cat2, use_smote, True), X2_train, X2_test, cv=(np.nan, np.nan), note="no unregistration features")

[  2150s] C0 clean: author XGBoost, author features minus unregistration CV acc nan±nan | TEST acc 0.9506  f1 0.9490  auc 0.9875  (9s)
[  2189s] C1 clean: best improved model minus unregistration CV acc nan±nan | TEST acc 0.9617  f1 0.9602  auc 0.9910  (38s)


Pipeline(steps=[('preprocess',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(add_indicator=True,
                                                                                 strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['date_registration',
                                                   'num_of_prev_attempts',
                                                   'studied_credits',
                                                   'module_presentation_length',
                                                   'num_assessments',
                                                   'mean_score', 'max_score',
                                                   'min_score', 'std_score',
                                                   'total_weight...
                               feature_types=None, feature_weights=None,
                               gamma=0, grow_policy=None, importance_type=None,
                               interaction_constraints=None, learning_rate=0.02,
                               max_bin=None, max_cat_threshold=None,
                               max_cat_to_onehot=None, max_delta_step=None,
                               max_depth=8, max_leaves=None, min_child_weight=3,
                               missing=nan, monotone_constraints=None,
                               multi_strategy=None, n_estimators=800, n_jobs=-1,
                               num_parallel_tree=None, ...))])

## 6. Comparison with the author's baseline, figure and summary
The last line printed starts with `FINAL:` and states the gain in percentage points. If the gain is below 1 point, report it as it is.

In [ ]:
# =====================================================================================================
# PART 4: statistics against the author's baseline, figure, summary
# =====================================================================================================
def paired_bootstrap(yhat_a, yhat_b, n=2000, seed=0):
    rng = np.random.RandomState(seed)
    yt = y_test.values
    ca, cb = (yhat_a == yt).astype(int), (yhat_b == yt).astype(int)
    d = []
    for _ in range(n):
        i = rng.randint(0, len(yt), len(yt))
        d.append(cb[i].mean() - ca[i].mean())
    return float(np.mean(cb) - np.mean(ca)), [float(np.percentile(d, 2.5)), float(np.percentile(d, 97.5))]

def mcnemar(yhat_a, yhat_b):
    yt = y_test.values
    ca, cb = yhat_a == yt, yhat_b == yt
    b, c = int((ca & ~cb).sum()), int((~ca & cb).sum())
    if b + c == 0:
        return 1.0, b, c
    try:
        pv = stats.binomtest(min(b, c), b + c, 0.5).pvalue
    except AttributeError:
        pv = stats.binom_test(min(b, c), b + c, 0.5)
    return float(pv), b, c

base_name = results[0]["config"]
base_hat = test_preds[base_name][0]
rows = []
for r in results[1:]:
    h = test_preds[r["config"]][0]
    diff, ci = paired_bootstrap(base_hat, h)
    pv, b, c = mcnemar(base_hat, h)
    rows.append({"config": r["config"], "acc_gain_vs_B0": diff, "gain_ci_lo": ci[0], "gain_ci_hi": ci[1],
                 "mcnemar_p": pv, "fixed_by_new": c, "broken_by_new": b})
cmp_df = pd.DataFrame(rows)
cmp_df.to_csv(os.path.join(ARGS.out, "comparison_vs_author.csv"), index=False)
res_df = pd.DataFrame(results)
res_df.to_csv(os.path.join(ARGS.out, "improvement_results.csv"), index=False)

fig, ax = plt.subplots(figsize=(9.5, 4.6))
labels = [r["config"].split(":")[0].split(" ")[0] + " " + (r["config"].split(" ", 1)[1][:34] if " " in r["config"] else "") for r in results]
vals = [r["test_acc"] * 100 for r in results]
ax.barh(range(len(vals)), vals, color=["#6B7280" if i == 0 else ("#B45309" if results[i]["config"].startswith("C") else "#1E3A8A") for i in range(len(vals))])
for i, v in enumerate(vals):
    ax.text(v + 0.03, i, f"{v:.2f}%", va="center", fontsize=8)
b0 = results[0]["test_acc"] * 100
ax.axvline(b0, color="#6B7280", ls="--", lw=1); ax.axvline(b0 + 1, color="#0F766E", ls="--", lw=1)
ax.text(b0 + 1, -0.9, "+1 point", color="#0F766E", fontsize=8, ha="center")
ax.set_yticks(range(len(vals))); ax.set_yticklabels(labels, fontsize=7); ax.invert_yaxis()
ax.set_xlim(min(vals) - 1, max(max(vals), b0 + 1) + 0.8); ax.set_xlabel("Test accuracy (%) on the author's 20% split (grey = author baseline, amber = without unregistration features)")
for s in ("top", "right"): ax.spines[s].set_visible(False)
fig.tight_layout(); fig.savefig(os.path.join(ARGS.out, "fig_improvement.png"), dpi=200); plt.close(fig)

final = results[[r["config"].startswith("F4") for r in results].index(True)]
cm = confusion_matrix(y_test, test_preds[final["config"]][0])
summary = {"author_baseline_test_acc": results[0]["test_acc"], "final_config": final["config"], "final_test_acc": final["test_acc"],
           "gain_points": (final["test_acc"] - results[0]["test_acc"]) * 100,
           "gain_vs_B0": cmp_df[cmp_df.config == final["config"]].to_dict("records"), "final_confusion_matrix": cm.tolist(),
           "threshold": thr, "n_extra_features": len(extra_cols), "selected_by": "training-set CV accuracy (test set not used for any choice)",
           "reached_plus_1_point": bool(final["test_acc"] - results[0]["test_acc"] >= 0.01),
           "libraries": {"xgboost": XGBClassifier is not None, "lightgbm": LGBMClassifier is not None, "catboost": CatBoostClassifier is not None, "smote": SMOTE is not None},
           "runtime_minutes": (time.time() - T0) / 60}
json.dump(summary, open(os.path.join(ARGS.out, "improvement_summary.json"), "w"), indent=2, default=lambda o: o.item() if hasattr(o, "item") else str(o))
print("\n" + "=" * 100)
print(res_df[["config", "cv_acc", "test_acc", "test_f1", "test_auc", "test_errors"]].round(4).to_string(index=False))
print("\nCompared with the author's baseline (paired, same test students):")
print(cmp_df.round(4).to_string(index=False))
print(f"\nFINAL: {final['config']}  test accuracy {final['test_acc']*100:.2f}%  vs author baseline {results[0]['test_acc']*100:.2f}%  "
      f"=> {summary['gain_points']:+.2f} points   (+1 point reached: {summary['reached_plus_1_point']})")
log(f"DONE in {(time.time()-T0)/60:.1f} min. Results saved in {ARGS.out}")


                                                        config  cv_acc  test_acc  test_f1  test_auc  test_errors
                  B0 author: XGBoost + SMOTE (author features)  0.9513    0.9529   0.9514    0.9883          307
                                  B1 author features, no SMOTE  0.9506    0.9524   0.9508    0.9883          310
                     F1 + extra features (author XGB settings)  0.9589    0.9610   0.9596    0.9912          254
                           F2 + extra features + tuned XGBoost  0.9593    0.9615   0.9600    0.9912          251
  F3 + extra features + soft-voting ensemble (gbm+lgbm+cat+rf)  0.9592    0.9626   0.9612    0.9913          244
                           F4 = F2 + OOF-chosen threshold 0.50  0.9593    0.9615   0.9600    0.9912          251
C0 clean: author XGBoost, author features minus unregistration     NaN    0.9506   0.9490    0.9875          322
            C1 clean: best improved model minus unregistration     NaN    0.9617   0.9602    0.

## 7. Save and download
Results are already saved in the `out` folder on Drive (`improvement_results.csv`, `comparison_vs_author.csv`, `improvement_summary.json`, `best_params.json`, `extra_features.csv`, `fig_improvement.png`).
To send me the notebook with its outputs: **File → Download → Download .ipynb**.